# RAG: Baseline vs Re-Ranking

**Document:** ICC Men's T20 World Cup 2026 Playing Conditions (106 pages, Feb 2026)

- **Pipeline 1:** bi-encoder → top 10 by cosine → top 3 → LLM → Answer 1
- **Pipeline 2:** same top 10 → cross-encoder re-rank → top 3 → LLM → Answer 2

Chunks come in two versions with the same boundaries: **A** = plain text, **B** = section title + text. Both pipelines are run on both.

## Setup

In [1]:
import os, re, random, hashlib, urllib.request
from pathlib import Path

ROOT = Path.cwd()
DATA, OUT = ROOT / "data", ROOT / "outputs"
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
os.environ.setdefault("HF_HOME", str(ROOT / "models" / "hf"))  # keep model downloads in the project
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

import pymupdf
import pandas as pd
from transformers import AutoTokenizer

pd.set_option("display.max_colwidth", 140)
random.seed(0)

## Phase 1: Chunking

### 1.1 Load PDF
Not committed (ICC copyright), so it's downloaded from the ICC site if missing.

In [2]:
PDF_URL = "https://images.icc-cricket.com/image/upload/prd/yyt7m8uh9c1uehrbvuwb.pdf"
PDF_PATH = DATA / "icc_t20wc_2026_playing_conditions.pdf"
PDF_SHA256 = "0a5af6c6bd1677c835a4b7a0c81ebb68a1b0894c87ef0f158348dc84e58f323b"

if not PDF_PATH.exists():
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
sha = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()
assert sha == PDF_SHA256, f"PDF differs from the version used in this study: {sha}"

doc = pymupdf.open(PDF_PATH)
print(f"{PDF_PATH.name}: {doc.page_count} pages, {PDF_PATH.stat().st_size / 1e6:.1f} MB, sha256 OK")

icc_t20wc_2026_playing_conditions.pdf: 106 pages, 0.8 MB, sha256 OK


### 1.2 Raw text
Each page starts with a page number and a running header. Clause numbers are followed by a tab (`9.1\t`), and section titles are in caps.

In [3]:
print(repr(doc[15].get_text()[:700]))

"16\nPlaying Conditions  |  ICC Men's T20 World Cup 2026\n9\t\nPREPARATION AND MAINTENANCE OF \nTHE PLAYING AREA\n9.1\t\nRolling\nThe pitch shall not be rolled during the match except as permitted in clauses 9.1.1 and\xa09.1.2.\n9.1.1\t\nFrequency and duration of rolling During the match the pitch may be rolled at the \nrequest of the captain of the side batting second, for a period of not more than \n7\xa0minutes, before the start of the second innings.\n9.1.2\t\nRolling after a delayed start In addition to the rolling permitted above, if, after the toss \nand before the first innings of the match, the start is delayed, the captain of the batting \nside may request that the pitch be rolled for not more than 7 minute"


### 1.3 Clean
Skip the cover (p.1) and the contents pages (p.3, p.72). Drop page numbers, headers and the blank form lines on p.99.

In [4]:
SKIP_PAGES = {1, 3, 72}   # cover, contents, appendix contents
HEADER_RE = re.compile(r"^\s*(\d{1,3}\s+)?(ICC Men's T20 World Cup 2026\s*\|\s*Playing Conditions"
                       r"|Playing Conditions\s*\|\s*ICC Men's T20 World Cup 2026)\s*$")

def extract_lines(doc):
    lines = []                                   # (page_number, line)
    for pno, page in enumerate(doc, start=1):
        if pno in SKIP_PAGES:
            continue
        raw = page.get_text().replace("\xa0", " ").replace("\x07", "")
        for i, line in enumerate(raw.splitlines()):
            if i == 0 and line.strip() == str(pno):          # page number
                continue
            if HEADER_RE.match(line) or not line.strip() or re.search(r"_{3,}", line):
                continue
            lines.append((pno, line.rstrip(" ")))    # keep TABs: they mark clause numbers
    return lines

lines = extract_lines(doc)
print(f"{len(lines)} lines kept from {doc.page_count - len(SKIP_PAGES)} pages")

5151 lines kept from 103 pages


### 1.4 Parse clauses
Walk through the lines, keeping track of the current appendix and section. Every clause number starts a new unit.

In [5]:
CLAUSE_RE = re.compile(r"^(\d{1,2}(?:\.\d{1,2})+|\d{1,2}\.)(?:\t\s*(.*)|\s*)$")
SECTION_RE = re.compile(r"^(\d{1,2})(?:\t\s*(.*)|\s*)$")
APPENDIX_RE = re.compile(r"^APPENDIX ([A-J]):\s*(.*)$")

def is_caps(line):
    letters = [c for c in line if c.isalpha()]
    return len(letters) >= 3 and all(c.isupper() for c in letters)

def read_caps_title(lines, j, first, stop_re):
    title = [first]
    while (j < len(lines) and is_caps(lines[j][1]) and not stop_re.match(lines[j][1])
           and not APPENDIX_RE.match(lines[j][1].strip())):
        title.append(lines[j][1].strip()); j += 1
    return " ".join(t for t in title if t), j

def parse_units(lines):
    units, appendix, section, cur = [], None, ("0", "PREAMBLE"), None
    def new(clause, pno, first=""):
        return {"appendix": appendix, "section": section, "clause": clause, "page": pno,
                "lines": [first] if first.strip() else []}
    i = 0
    while i < len(lines):
        pno, line = lines[i]
        nxt = lines[i + 1][1] if i + 1 < len(lines) else ""
        m_app, m_sec, m_cl = APPENDIX_RE.match(line.strip()), SECTION_RE.match(line), CLAUSE_RE.match(line)
        if m_app:
            units.append(cur)
            title, i = read_caps_title(lines, i + 1, m_app.group(2).strip(), SECTION_RE)
            appendix = (m_app.group(1), title)
            section = appendix
            cur = new(None, pno)
        elif m_sec and not m_cl and (is_caps(m_sec.group(2) or "") or (not (m_sec.group(2) or "").strip() and is_caps(nxt))):
            units.append(cur)
            title, i = read_caps_title(lines, i + 1, (m_sec.group(2) or "").strip(), CLAUSE_RE)
            section = (m_sec.group(1), title)
            cur = new(None, pno)
        elif m_cl:
            units.append(cur)
            cur = new(m_cl.group(1).rstrip("."), pno, m_cl.group(2) or "")
            i += 1
        else:
            cur = cur or new(None, pno)               # preamble before the first section
            cur["lines"].append(line)
            i += 1
    units.append(cur)
    return [u for u in units if u and u["lines"]]

def join_lines(ls):
    t = "\n".join(l.strip() for l in ls if l.strip())
    t = re.sub(r"-\n(?=[a-z])", "-", t)              # "wicket-\nkeeper" -> "wicket-keeper"
    return re.sub(r"\s+", " ", t.replace("\t", " ").replace("\n", " ")).strip()

units = parse_units(lines)
for u in units:
    body = join_lines(u["lines"])
    u["text"] = f"{u['clause']} {body}" if u["clause"] else body
print(f"{len(units)} clause units")
pd.DataFrame([{"clause": u["clause"], "section": " ".join(u["section"]), "page": u["page"], "text": u["text"][:100]}
              for u in units[60:66]])

1216 clause units


,clause,section,page,text
0,2.7.1,2 THE UMPIRES,7,"2.7.1 It is solely for the umpires together to decide whether either conditions of ground, weather o"
1,2.7.2,2 THE UMPIRES,7,2.7.2 Conditions shall be regarded as dangerous if there is actual and foreseeable risk to the safet
2,2.7.3,2 THE UMPIRES,7,"2.7.3 Conditions shall be regarded as unreasonable if, although posing no risk to safety, it would n"
3,2.7.4,2 THE UMPIRES,7,2.7.4 If the umpires consider the ground is so wet or slippery as to deprive the bowler of a reasona
4,2.8,2 THE UMPIRES,7,2.8 Suspension of play in dangerous or unreasonable circumstances
5,2.8.1,2 THE UMPIRES,7,2.8.1 All references to ground include the pitch. See clause 6.1 (Area of pitch).


### 1.5 Chunk
- Merge consecutive clauses from the same section, up to ~180 tokens (MiniLM's limit is 256).
- Split clauses over 220 tokens by sentence, with 1 sentence of overlap.
- Never leave a heading alone at the end of a chunk.

Chunks are kept small on purpose: with big chunks the answer is almost always in the top 3 already, and re-ranking has nothing to fix.

In [6]:
TARGET, MAX = 180, 220
tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
ntok = lambda s: len(tok.tokenize(s))
for u in units:
    u["ntok"] = ntok(u["text"])

def split_sentences(text):
    sents = []
    for s in re.split(r"(?<=[.;:])\s+(?=[A-Z0-9(])", text):
        words = s.split()
        while ntok(" ".join(words)) > TARGET:    # very long sentence
            k = len(words) // 2
            while ntok(" ".join(words[:k])) > TARGET: k //= 2
            while k < len(words) and ntok(" ".join(words[:k + 10])) <= TARGET: k += 10
            sents.append(" ".join(words[:k])); words = words[k:]
        sents.append(" ".join(words))
    return sents

def split_long(u):
    out, win = [], []
    for s in split_sentences(u["text"]):
        if win and ntok(" ".join(win + [s])) > TARGET:
            out.append(" ".join(win))
            win = [win[-1]] if ntok(win[-1]) < TARGET // 3 else []
        win.append(s)
    out.append(" ".join(win))
    return [dict(u, text=t, ntok=ntok(t)) for t in out]

def pack(units):
    pieces = [p for u in units for p in (split_long(u) if u["ntok"] > MAX else [u])]
    chunks, buf = [], []
    def emit():
        clauses = [b["clause"] for b in buf if b["clause"]]
        chunks.append({"appendix": buf[0]["appendix"], "section": buf[0]["section"], "page": buf[0]["page"],
                       "clauses": (f"{clauses[0]}–{clauses[-1]}" if len(set(clauses)) > 1 else (clauses[0] if clauses else "")),
                       "text": " ".join(b["text"] for b in buf)})
    for p in pieces:
        same = buf and (p["appendix"], p["section"]) == (buf[0]["appendix"], buf[0]["section"])
        size = sum(b["ntok"] for b in buf)
        if buf and not (same and size < 15) and (not same or size + p["ntok"] > TARGET):
            carry = [buf.pop()] if same and len(buf) > 1 and buf[-1]["ntok"] < 15 else []   # bare heading
            emit(); buf = carry
        buf.append(p)
    emit()
    return chunks

chunks = pack(units)
print(f"{len(units)} clause units -> {len(chunks)} chunks")

1216 clause units -> 445 chunks


### 1.6 Two versions
`text` (A) and `text_titled` (B = `[section] ` + text). Same IDs in both.

In [7]:
SMALL = {"and", "of", "the", "for", "to", "on", "in", "a", "an", "or", "by"}
def nice(s):
    return " ".join(w if w.startswith("(") and w.strip("()").isupper() and len(w.strip("()")) <= 4
                    else w.lower() if k and w.lower() in SMALL else w.capitalize()
                    for k, w in enumerate(s.split()))

for k, c in enumerate(chunks):
    c["id"] = f"c{k:03d}"
    sec_num, sec_title = c["section"]
    path = f"Appendix {c['appendix'][0]}: {nice(c['appendix'][1])}" if c["appendix"] else ""
    if not c["appendix"] or sec_title != c["appendix"][1]:
        path = (path + " > " if path else "") + (nice(sec_title) if sec_num == "0" else f"{sec_num} {nice(sec_title)}")
    c["section_path"] = path
    c["text_titled"] = f"[{path}] {c['text']}"
    c["ntok"], c["ntok_titled"] = ntok(c["text"]), ntok(c["text_titled"])

df = pd.DataFrame(chunks)[["id", "section_path", "clauses", "page", "ntok", "ntok_titled", "text", "text_titled"]]
df.head(3)

,id,section_path,clauses,page,ntok,ntok_titled,text,text_titled
0,c000,Preamble,,2,165,170,PREAMBLE – THE SPIRIT OF CRICKET Cricket owes much of its appeal and enjoyment to the fact that it should be played not only according t...,[Preamble] PREAMBLE – THE SPIRIT OF CRICKET Cricket owes much of its appeal and enjoyment to the fact that it should be played not only ...
1,c001,Preamble,,2,81,86,"Congratulate the opposition on their successes, and enjoy those of your own team. Thank the officials and your opposition at the end of ...","[Preamble] Congratulate the opposition on their successes, and enjoy those of your own team. Thank the officials and your opposition at ..."
2,c002,1 The Players,1.1–1.2.1,4,136,141,"1.1 Number of players A match is played between two sides, each of eleven players, one of whom shall be captain. 1.2 Nomination and repl...","[1 The Players] 1.1 Number of players A match is played between two sides, each of eleven players, one of whom shall be captain. 1.2 Nom..."


### 1.7 Save

In [8]:
CHUNKS_PATH = DATA / "chunks.jsonl"
df.to_json(CHUNKS_PATH, orient="records", lines=True, force_ascii=False)
print(f"saved {len(df)} chunks -> {CHUNKS_PATH.relative_to(ROOT)}")

saved 445 chunks -> data/chunks.jsonl


### 1.8 Sanity checks

In [9]:
print(df[["ntok", "ntok_titled"]].describe().round(1).T[["count", "min", "25%", "50%", "75%", "max"]])
print(f"\nchunks over 256 tokens: plain {(df.ntok > 256).sum()}, titled {(df.ntok_titled > 256).sum()}")
print(f"main-body chunks: {(~df.section_path.str.startswith('Appendix')).sum()}, appendix chunks: {df.section_path.str.startswith('Appendix').sum()}")

             count   min    25%    50%    75%    max
ntok         445.0   4.0  115.0  148.0  168.0  215.0
ntok_titled  445.0  11.0  122.0  155.0  175.0  223.0

chunks over 256 tokens: plain 0, titled 0
main-body chunks: 322, appendix chunks: 123


In [10]:
# chunks per part
part = df.section_path.where(df.section_path.str.startswith("Appendix"), "Main body (sections 1-42)").str.split(" > ").str[0]
part.value_counts().sort_index().to_frame("chunks")

,chunks
section_path,
Appendix A: Definitions,28
Appendix B: Equipment,13
Appendix C: The Venue,4
Appendix D: Decision Review System (DRS),55
Appendix E: Calculations,5
Appendix F: Super Over Procedure,9
Appendix G: Application of the Reserve Day for the Semi-finals and Final,4
Appendix H: Use of Electronic Communications Equipment,1
Appendix I: On-field Injury Assessments and Treatment,1


In [11]:
# a few random chunks
for _, r in df.sample(3, random_state=1).iterrows():
    print(f"{r.id} | {r.section_path} | clauses {r.clauses} | p.{r.page} | {r.ntok} tokens\n{r.text}\n")

c249 | 36 Leg Before Wicket | clauses 36.1–36.1.4 | p.56 | 176 tokens
36.1 Out LBW The striker is out LBW if all the circumstances set out in clauses 36.1.1 to 36.1.5 apply. 36.1.1 The bowler delivers a ball, not being a No ball 36.1.2 the ball, if it is not intercepted full-pitch, pitches in line between wicket and wicket or on the off side of the striker’s wicket 36.1.3 the ball not having previously touched his bat, the striker intercepts the ball, either full-pitch or after pitching, with any part of his person 36.1.4 the point of impact, even if above the level of the bails, either is between wicket and wicket or if the striker has made no genuine attempt to play the ball with the bat, is between wicket and wicket or outside the line of the off stump.

c180 | 22 Wide Ball | clauses 22.9 | p.40 | 58 tokens
22.9 Out from a Wide When Wide ball has been called, neither batter shall be out under any of the Playing Conditions except clause 35 (Hit wicket), clause 37 (Obstructing the fie

Some chunks to keep in mind for later:
- 9.1: the rolling rule stays together with its exceptions
- Follow-on: the whole chunk is "Shall not apply.", so it can only be found in version B
- Main rules say 11 players, warm-ups (App. J) say max 15 → keyword trap

In [12]:
show = lambda mask: df[mask][["id", "section_path", "clauses", "page", "text"]]
pd.concat([
    show(df.text.str.contains("shall not be rolled")),
    show(df.text.str.fullmatch("Shall not apply.")),
    show(df.text.str.startswith("1.1 Number of players") | df.text.str.contains("Each side shall consist of a maximum 15 players")),
])

,id,section_path,clauses,page,text
53,c053,9 Preparation and Maintenance of the Playing Area,9.1–9.1.2,16,9.1 Rolling The pitch shall not be rolled during the match except as permitted in clauses 9.1.1 and 9.1.2. 9.1.1 Frequency and duration ...
91,c091,14 The Follow-on,,24,Shall not apply.
92,c092,15 Declaration and Forfeiture,,24,Shall not apply.
2,c002,1 The Players,1.1–1.2.1,4,"1.1 Number of players A match is played between two sides, each of eleven players, one of whom shall be captain. 1.2 Nomination and repl..."
442,c442,Appendix J: Summary of Changes to Playing Conditions for Warm-up Matches,1.1,105,NOTE: ICC Men’s T20 World Cup 2026 warm-up matches will not be official Twenty 20 Internationals. ICC Match Referees will not be on duty...


In [13]:
r = df[df.section_path.str.contains("Follow-on")].iloc[0]
print("A:", r.text)
print("B:", r.text_titled)

A: Shall not apply.
B: [14 The Follow-on] Shall not apply.
